<a href="https://colab.research.google.com/github/Pratyakshk05/deep-learning/blob/main/lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader, random_split
import wandb
import matplotlib.pyplot as plt

# ---------------------------
# CONFIG
# ---------------------------
BATCH_SIZE = 128
EPOCHS = 10
LR = 0.001
PATCH_SIZE = 4
EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 6
NUM_CLASSES = 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

wandb.init(project="ViT-vs-ResNet")

# ---------------------------
# DATA (FIXED NORMALIZATION)
# ---------------------------
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                       download=True, transform=transform_aug)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

# ---------------------------
# VISION TRANSFORMER
# ---------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=128):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim,
                             kernel_size=patch_size,
                             stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class ViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = PatchEmbedding()

        self.cls_token = nn.Parameter(torch.randn(1, 1, EMBED_DIM))
        self.pos_embed = nn.Parameter(torch.randn(1, 65, EMBED_DIM))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM,
            nhead=NUM_HEADS,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=NUM_LAYERS)

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(EMBED_DIM),
            nn.Linear(EMBED_DIM, NUM_CLASSES)
        )

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed[:, :x.size(1)]
        x = self.transformer(x)

        cls_output = x[:, 0]
        return self.mlp_head(cls_output)


# ---------------------------
# RESNET-18
# ---------------------------
def get_resnet():
    model = resnet18(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


# ---------------------------
# LOSS FUNCTIONS
# ---------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()


def get_loss(name):
    if name == "ce":
        return nn.CrossEntropyLoss()
    elif name == "label_smooth":
        return nn.CrossEntropyLoss(label_smoothing=0.1)
    elif name == "focal":
        return FocalLoss()


# ---------------------------
# OPTIMIZER
# ---------------------------
def get_optimizer(name, model):
    if name == "sgd":
        return optim.SGD(model.parameters(), lr=LR, momentum=0.9)
    elif name == "adam":
        return optim.Adam(model.parameters(), lr=LR)
    elif name == "rmsprop":
        return optim.RMSprop(model.parameters(), lr=LR)


# ---------------------------
# TRAIN FUNCTION
# ---------------------------
def train(model, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (out.argmax(1) == y).sum().item()

    acc = correct / len(train_loader.dataset)
    return total_loss, acc


# ---------------------------
# EVAL FUNCTION
# ---------------------------
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)

            total_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss, acc


# ---------------------------
# RUN EXPERIMENT
# ---------------------------
def run_experiment(model_name="vit", loss_name="ce", opt_name="adam"):
    if model_name == "vit":
        model = ViT().to(DEVICE)
    else:
        model = get_resnet().to(DEVICE)

    criterion = get_loss(loss_name)
    optimizer = get_optimizer(opt_name, model)

    for epoch in range(EPOCHS):
        train_loss, train_acc = train(model, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        print(f"{epoch}: Train Acc={train_acc:.3f}, Val Acc={val_acc:.3f}")

    test_loss, test_acc = evaluate(model, test_loader, criterion)
    print("Test Accuracy:", test_acc)

    return test_acc   # IMPORTANT


# ---------------------------
# RUN ALL CONFIGS + STORE BEST
# ---------------------------
configs = [
    ("vit", "ce", "adam"),
    ("vit", "focal", "sgd"),
    ("resnet", "ce", "adam"),
    ("resnet", "label_smooth", "rmsprop")
]

best_results = {"vit": 0, "resnet": 0}

for cfg in configs:
    print("Running:", cfg)
    acc = run_experiment(*cfg)

    model_name = cfg[0]
    if acc > best_results[model_name]:
        best_results[model_name] = acc


# ---------------------------
# FINAL PLOT (ONLY REQUIRED)
# ---------------------------
def plot_vit_vs_resnet(results):
    vit_acc = results["vit"]
    resnet_acc = results["resnet"]

    models = ["ViT", "ResNet-18"]
    acc = [vit_acc, resnet_acc]

    plt.figure()
    plt.bar(models, acc)
    plt.xlabel("Model")
    plt.ylabel("Test Accuracy")
    plt.title("ViT vs ResNet-18 Comparison")
    plt.show()


plot_vit_vs_resnet(best_results)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pratyakshk05 (pratyakshk05-delhi-technological-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 170M/170M [00:07<00:00, 24.0MB/s]


Running: ('vit', 'ce', 'adam')
0: Train Acc=0.279, Val Acc=0.347
1: Train Acc=0.390, Val Acc=0.431
2: Train Acc=0.449, Val Acc=0.447
3: Train Acc=0.477, Val Acc=0.505
4: Train Acc=0.498, Val Acc=0.520
5: Train Acc=0.522, Val Acc=0.519
6: Train Acc=0.529, Val Acc=0.541
7: Train Acc=0.545, Val Acc=0.544
8: Train Acc=0.554, Val Acc=0.567
9: Train Acc=0.571, Val Acc=0.576
Test Accuracy: 0.5576
Running: ('vit', 'focal', 'sgd')
0: Train Acc=0.194, Val Acc=0.238
1: Train Acc=0.232, Val Acc=0.256
2: Train Acc=0.246, Val Acc=0.277
3: Train Acc=0.266, Val Acc=0.289
4: Train Acc=0.281, Val Acc=0.301
5: Train Acc=0.293, Val Acc=0.304
